# Setup

To run this notebook on Colab, a few setup steps are required. Follow along step by step:

1. **Clone the `dlfb` library**  
   First, clone the repository that contains the `dlfb` library.

In [ ]:
%cd /content
!rm -rf ./dlfb-pytorch-clone/
!git clone "https://github.com/deep-learning-for-biology/deep-learning-for-biology-pytorch.git" dlfb-pytorch-clone --branch main
%cd dlfb-pytorch-clone

2. **Install dependencies**  
   Once the library is cloned, install the required dependencies.

In [ ]:
!pip install -e '.[dna]'

3. **Providion the datasets**  
   You’ll then need to access and download the necessary datasets for this chapter.

In [ ]:
# NOTE: exclude models with '--no-models' flag
!dlfb-provision --chapter dna

4. **Load the `dlfb` package**  
   Finally, load the `dlfb` package.  
   - ⚠️ Note: Loading can sometimes be finicky. If you encounter issues, simply **restart the runtime**. All previously downloaded data and installed packages will persist, so you can re-run the load step without repeating everything.

In [ ]:
try:
  import dlfb_pytorch
except ImportError as exc:
  import site
  site.main()
  import dlfb_pytorch

from dlfb_pytorch.utils.display import display

# 3. Learning the Logic of DNA


## 3.1. Biology Primer
### 3.1.1. What Exactly Is DNA?
### 3.1.2. Coding and Noncoding Regions
### 3.1.3. How Transcription Factors Direct Gene Activity
### 3.1.4. Measuring Where Transcription Factors Bind


## 3.2. Machine Learning Primer
### 3.2.1. Convolutional Neural Networks
### 3.2.2. Convolutions for DNA Sequences
### 3.2.3. Transformers
### 3.2.4. Attention
### 3.2.5. Query, Key and Value Intuition
### 3.2.6. Multiheaded Attention
### 3.2.7. Representing Positional Information
### 3.2.8. Model Interpretation
### 3.2.9. In Silico Saturation Mutagenesis
### 3.2.10. Input Gradients


In [ ]:
import torch
import torch.nn as nn
import numpy as np

## 3.3. Building a Simple Prototype
### 3.3.1. Building a Dataset
#### 3.3.1.1. Loading the Labeled Sequences


In [ ]:
import pandas as pd

from dlfb_pytorch.utils.context import assets

train_df = pd.read_csv(assets("dna/datasets/CTCF_train_sequences.csv"))
print(train_df)

In [ ]:
train_df["label"].value_counts()

In [ ]:
from dlfb_pytorch.dna.utils import dna_to_one_hot

display(["import numpy as np", dna_to_one_hot])

In [ ]:
dna_to_one_hot("AAACGT")

In [ ]:
x_train = np.array([dna_to_one_hot(seq) for seq in train_df["sequence"]])
y_train = train_df["label"].values[:, None]

In [ ]:
from dlfb_pytorch.dna.dataset import load_dataset

display([load_dataset])

#### 3.3.1.2. Convert the Data to a TensorFlow Dataset


In [ ]:
from dlfb_pytorch.dna.dataset import create_dataloader

display([create_dataloader])

In [ ]:
batch_size = 32

train_dl = create_dataloader(
  load_dataset(assets("dna/datasets/CTCF_train_sequences.csv")),
  batch_size=batch_size,
  is_training=True,
)

In [ ]:
batch = next(train_ds.as_numpy_iterator())
print(f'Batch sequence shape: {batch["sequences"].shape}')
print(f'Batch sequence instances: {batch["sequences"][:3,:3,]}...')
print(f'Batch labels shape: {batch["labels"].shape}')
print(f'Batch labels instances: {batch["labels"][:3,]}...')

In [ ]:
valid_ds = load_dataset(assets("dna/datasets/CTCF_valid_sequences.csv"))

### 3.3.2. Defining a Simple Convolutional Model


In [ ]:
from dlfb_pytorch.dna.model import ConvModel

display([ConvModel])

In [ ]:
model = ConvModel()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seq_len = batch["sequences"].shape[1] if isinstance(batch, dict) else batch[0].shape[1]
model = ConvModel(seq_len=seq_len).to(device)

dummy_input = torch.ones(1, seq_len, 4, device=device)
print(dummy_input.shape)

#### 3.3.2.1. Examining Model Tensor Shapes


In [ ]:
params.keys()

In [ ]:
for layer_name in params.keys():
  print(f'Layer {layer_name} param shape: {params[layer_name]["kernel"].shape}')

#### 3.3.2.2. Making Predictions with the Model


In [ ]:
model.eval()
with torch.no_grad():
  seq, lab = next(iter(train_dl))
  logits = model(seq.to(device))

# Apply sigmoid to convert logits to probabilities.
probs = torch.sigmoid(logits)

# Print just the first few predictions.
print(probs[0:5])

#### 3.3.2.3. Defining a Loss Function


In [ ]:
import torch.nn.functional as F


def calculate_loss(model, sequences, labels):
  """Make predictions on batch and compute binary cross entropy loss."""
  logits = model(sequences)
  loss = F.binary_cross_entropy_with_logits(logits, labels)
  return loss

In [ ]:
calculate_loss(params, batch)

#### 3.3.2.4. Defining the `TrainState`


In [ ]:
learning_rate = 0.001

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# In PyTorch, model and optimizer are used directly.
# No TrainState wrapper needed.

In [ ]:
def create_train_state(model, rng, dummy_input, tx) -> TrainState:
  variables = model.init(rng, dummy_input)
  state = TrainState.create(
    apply_fn=model.apply, params=variables["params"], tx=tx
  )
  return state

#### 3.3.2.5. Defining a Single Training Step


In [ ]:
def train_step(model, optimizer, sequences, labels):
  """Run single training step to compute gradients and update model params."""
  model.train()
  sequences, labels = sequences.to(device), labels.to(device)
  optimizer.zero_grad()
  loss = calculate_loss(model, sequences, labels)
  loss.backward()
  optimizer.step()
  return loss.item()

In [ ]:
state, loss = train_step(state, batch)

In [ ]:
model.eval()
with torch.no_grad():
  seq, lab = next(iter(train_dl))
  calculate_loss(model, seq.to(device), lab.to(device))

#### 3.3.2.6. Training the Simple Model


In [ ]:
import tqdm

# Reinitialize model and optimizer for a fresh start.
model = ConvModel(seq_len=seq_len).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_losses, valid_losses = [], []
train_iter = iter(train_dl)

for step in tqdm.tqdm(range(500)):
  try:
    sequences, labels = next(train_iter)
  except StopIteration:
    train_iter = iter(train_dl)
    sequences, labels = next(train_iter)

  loss = train_step(model, optimizer, sequences, labels)
  train_losses.append({"step": step, "loss": loss})

  if step % 100 == 0:
    # Compute loss on entire validation set.
    model.eval()
    with torch.no_grad():
      for vseq, vlab in [next(iter(valid_dl))]:
        vseq, vlab = vseq.to(device), vlab.to(device)
        vloss = calculate_loss(model, vseq, vlab)
        valid_losses.append({"step": step, "loss": vloss.item()})

losses = pd.concat(
  [
    pd.DataFrame(train_losses).assign(split="train"),
    pd.DataFrame(valid_losses).assign(split="valid"),
  ]
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from dlfb_pytorch.utils.metric_plots import DEFAULT_SPLIT_COLORS

sns.lineplot(
  data=losses,
  x="step",
  y="loss",
  hue="split",
  style="split",
  palette=DEFAULT_SPLIT_COLORS
);

#### 3.3.2.7. Sanity Checking the Model


In [ ]:
ctcf_motif_dna = "CCACCAGGGGGCGC" * 14 + "AAAA"
print("Length of CTCF motif-filled DNA string:", len(ctcf_motif_dna))

# We add the None here as a batch axis, since our model expects batched input.
ctcf_input = dna_to_one_hot(ctcf_motif_dna)[None, :]
ctcf_input.shape

In [ ]:
model.eval()
with torch.no_grad():
  ctcf_tensor = torch.tensor(ctcf_input, dtype=torch.float32, device=device)
  torch.sigmoid(model(ctcf_tensor))

In [ ]:
random_dna_strings = [
  "A" * 200,
  "C" * 200,
  "G" * 200,
  "T" * 200,
  "ACGTACGT" * 25,
  "TCGATCGT" * 25,
  "TATACGCG" * 25,
  "CAGGCAGG" * 25,
]

probabilities = []

model.eval()
for random_dna_string in random_dna_strings:
  random_dna_input = torch.tensor(
    dna_to_one_hot(random_dna_string)[None, :], dtype=torch.float32, device=device
  )
  with torch.no_grad():
    probabilities.append(torch.sigmoid(model(random_dna_input))[0])

probabilities

In [ ]:
from dlfb_pytorch.utils.restore import store

store(assets("dna/models/prototype"), model, optimizer, metrics=losses.to_dict("records"))

## 3.4. Increasing Complexity
### 3.4.1. *In Silico* Mutagenesis


In [ ]:
# The first positive example of a sequence that binds the transcription factor.
first_positive_index = np.argmax(valid_ds["labels"].flatten() == 1)

original_sequence = valid_ds["sequences"][first_positive_index].copy()
print(f'This sequence has label: {valid_ds["labels"][4]}')

In [ ]:
model.eval()
with torch.no_grad():
  orig_tensor = torch.tensor(original_sequence[None, :], dtype=torch.float32, device=device)
  pred = torch.sigmoid(model(orig_tensor))
pred

In [ ]:
sequence = original_sequence.copy()
print(f"Original base at index 100: {sequence[100]}")

sequence[100] = np.array([0, 1, 0, 0])
print(f"Mutated base at index 100: {sequence[100]}")

In [ ]:
model.eval()
with torch.no_grad():
  seq_tensor = torch.tensor(sequence[None, :], dtype=torch.float32, device=device)
  pred_with_mutation = torch.sigmoid(model(seq_tensor))
pred_with_mutation

#### 3.4.1.1. Implementing *In Silico* Saturation Mutagenesis


In [ ]:
def generate_all_mutations(sequence: np.ndarray) -> np.ndarray:
  """Generate all possible single base mutations of a one-hot DNA sequence."""
  mutated_sequences = []
  for i in range(sequence.shape[0]):
    # At each position, one of the four 'mutations' is the original base (no-op).
    for j in range(4):
      mutated_sequence = sequence.copy()
      mutated_sequence[i] = np.zeros(4)
      mutated_sequence[i][j] = 1
      mutated_sequences.append(mutated_sequence)

  sequences = np.stack(mutated_sequences)
  return sequences


mutated_sequences = generate_all_mutations(sequence=original_sequence.copy())
print(f"Shape of mutated sequences: {mutated_sequences.shape}")

In [ ]:
model.eval()
with torch.no_grad():
  mut_tensor = torch.tensor(mutated_sequences, dtype=torch.float32, device=device)
  preds = torch.sigmoid(model(mut_tensor))

preds = preds.cpu().numpy().reshape((200, 4))

In [ ]:
plt.figure(figsize=(10, 2))
sns.heatmap(preds.T, cmap="viridis", yticklabels=["A", "C", "G", "T"])
plt.xlabel("Position in DNA sequence")
plt.ylabel("DNA Base");

In [ ]:
model.eval()
with torch.no_grad():
  orig_tensor = torch.tensor(original_sequence[None, :], dtype=torch.float32, device=device)
  baseline_pred = torch.sigmoid(model(orig_tensor)).cpu().numpy()
deltas = preds - baseline_pred

plt.figure(figsize=(10, 2))
sns.heatmap(deltas.T, center=0, cmap="viridis", yticklabels=["A", "C", "G", "T"])
plt.xlabel("Position in DNA sequence")
plt.ylabel("DNA Base");

In [ ]:
from dlfb_pytorch.dna.inspect import describe_change

display([describe_change])

In [ ]:
for i in range(4):
  print(describe_change((100, i), deltas, original_sequence))

In [ ]:
from dlfb_pytorch.dna.inspect import plot_binding_site

importance = np.sum(np.abs(deltas), axis=1)
plot_binding_site(
  panels={
    "tiles": {"label": "Deltas", "values": deltas},
    "line": {"label": "Importance", "values": importance},
  }
);

#### 3.4.1.2. Verifying Motif Presence


In [ ]:
from dlfb_pytorch.dna.utils import one_hot_to_dna

display([one_hot_to_dna])

In [ ]:
print(one_hot_to_dna(original_sequence)[0:25], "...")

In [ ]:
plot_binding_site(
  panels={
    "tiles": {"label": "Deltas", "values": deltas},
    "line": {"label": "Importance", "values": importance},
  },
  highlight=(92, 106),
);

#### 3.4.1.3. Implementing Input Gradients


In [ ]:
from dlfb_pytorch.dna.utils import compute_input_gradient

display([compute_input_gradient])

In [ ]:
input_gradient = compute_input_gradient(state, original_sequence)
input_gradient.shape

In [ ]:
importance = np.sum(np.abs(input_gradient), axis=1)
plot_binding_site(
  panels={
    "tiles": {"label": "Gradients", "values": input_gradient},
    "line": {"label": "Importance", "values": importance},
  },
);

In [ ]:
important_sequence = one_hot_to_dna(original_sequence)[90:110]
print("Central DNA sequence with high importance: ", important_sequence)

plt.figure(figsize=(10, 2))
sns.heatmap(
  input_gradient[90:110].T,
  cmap="viridis",
  center=0,
  xticklabels=important_sequence,
  yticklabels=["A", "C", "G", "T"],
)
plt.tight_layout();

In [ ]:
from dlfb_pytorch.dna.inspect import plot_10_gradients

plot_10_gradients(model, valid_ds, target_label=1);

In [ ]:
plot_10_gradients(state, valid_ds, target_label=0);

### 3.4.2. Modelling Multiple Transcription Factors
#### 3.4.2.1. Preparing a Multi-TF Dataset


In [ ]:
transcription_factors = [
  "ARID3",
  "ATF2",
  "BACH1",
  "CTCF",
  "ELK1",
  "GABPA",
  "MAX",
  "REST",
  "SRF",
  "ZNF24",
]

#### 3.4.2.2. Defining a More Complex Model


In [ ]:
import numpy as np

num_steps = 1000

# Cosine decay schedule (computed manually for visualization)
learning_rates = [
  0.001 * 0.5 * (1 + np.cos(np.pi * i / num_steps)) for i in range(num_steps)
]

plt.scatter(range(num_steps), learning_rates)
plt.title("Learning Rate over Steps")
plt.ylabel("Learning Rate")
plt.xlabel("Step");

In [ ]:
from dlfb_pytorch.dna.model import ConvModelV2

display([ConvModelV2])

In [ ]:
torch.manual_seed(42)
model_v2 = ConvModelV2(seq_len=seq_len).to(device)
optimizer_v2 = torch.optim.Adam(model_v2.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
  optimizer_v2, T_max=num_steps
)

In [ ]:
from dlfb_pytorch.dna.train import train_step as lib_train_step

display([lib_train_step])

In [ ]:
# Overfit on one batch.
sequences, labels = next(iter(train_dl))
for i in range(5):
  metrics = lib_train_step(model_v2, optimizer_v2, sequences, labels, device)
  print(f"Step {i} loss: {metrics['loss']}")

In [ ]:
from dlfb_pytorch.dna.train import compute_metrics, eval_step

display([eval_step, compute_metrics])

In [ ]:
# Evaluate the batch.
metrics = eval_step(state, batch)
print(metrics)

In [ ]:
from dlfb_pytorch.dna.train import train

display([train])

In [ ]:
from dlfb_pytorch.dna.dataset import load_dataset_splits

display([load_dataset_splits])

In [ ]:
prefix = assets("dna/datasets")
tf_metrics = {}

for transcription_factor in transcription_factors:
  dataset_splits = load_dataset_splits(
    assets("dna/datasets"), transcription_factor, batch_size
  )
  torch.manual_seed(42)

  # Determine sequence length from data.
  sample_seq, _ = next(iter(dataset_splits["train"]))
  tf_seq_len = sample_seq.shape[1]

  model_tf = ConvModelV2(seq_len=tf_seq_len).to(device)
  optimizer_tf = torch.optim.Adam(model_tf.parameters(), lr=learning_rate)

  _, _, metrics = train(
    model=model_tf,
    optimizer=optimizer_tf,
    dataset_splits=dataset_splits,
    num_steps=num_steps,
    eval_every=100,
    device=device,
    store_path=assets(f"dna/models/{transcription_factor}"),
  )

  tf_metrics.update({transcription_factor: metrics})

In [ ]:
from dlfb_pytorch.dna.inspect import plot_learning

tf = "CTCF"
plot_learning(tf_metrics[tf], tf);

In [ ]:
from dlfb_pytorch.utils.metric_plots import to_df

tf_df = []
for tf, metrics in tf_metrics.items():
  tf_df.append(to_df(metrics).assign(TF=tf))
tf_df = pd.concat(tf_df)

auc_df = tf_df[(tf_df["metric"] == "auc") & (tf_df["split"] == "valid")]
max_auc_by_tf = auc_df.groupby("TF")["mean"].max()
tf_order = max_auc_by_tf.sort_values(ascending=False).index.tolist()
tf_df["TF"] = pd.Categorical(tf_df["TF"], categories=tf_order, ordered=True)

In [ ]:
sns.set_context("notebook", font_scale=3, rc={"lines.linewidth": 2.5})
sns.set_style("ticks", {"axes.grid": True})
g = sns.relplot(
  data=tf_df,
  x="round",
  y="mean",
  hue="split",
  style="metric",
  kind="line",
  col="TF",
  col_order=tf_order,
  col_wrap=4,
  alpha=0.8,
  palette=DEFAULT_SPLIT_COLORS,
  dashes=True,
)
g.set_axis_labels("Step", "Value")
g.set(ylim=(0, 1));

In [ ]:
print(max_auc_by_tf.sort_values(ascending=False))

## 3.5. Advanced Techniques


In [ ]:
from dlfb_pytorch.dna.model import ConvBlock, MLPBlock

display([ConvBlock, MLPBlock])

In [ ]:
from dlfb_pytorch.dna.model import ConvTransformerModel

display([ConvTransformerModel])

### 3.5.1. Adding Self-attention and Transformer Blocks


In [ ]:
from dlfb_pytorch.dna.model import TransformerBlock

display([TransformerBlock])

### 3.5.2. Defining Various Model Architectures


In [ ]:
models = {
  # Our standard 2-layer CNN with dropout and MLP.
  "baseline": ConvTransformerModel(),
  # Ablations: Remove or reduce certain components.
  # Only a single convolutional block.
  "single_conv_only": ConvTransformerModel(
    num_conv_blocks=1, num_transformer_blocks=0, num_mlp_blocks=0
  ),
  # Reduced capacity by lowering conv filters.
  "fewer_conv_channels": ConvTransformerModel(conv_filters=8),
  # Drop the MLP layers to test if they help.
  "remove_MLP": ConvTransformerModel(num_mlp_blocks=0),
  # Potential improvements: Add more expressive capacity.
  # Add a transformer block after convolutions.
  "add_one_transformer_block": ConvTransformerModel(num_transformer_blocks=1),
  # Stack two transformer blocks.
  "add_two_transformer_block": ConvTransformerModel(num_transformer_blocks=2),
}

### 3.5.3. Sweeping Over the Different Models


In [ ]:
# Train and evaluate multiple model architectures on the ZNF24 dataset.
transcription_factor = "ZNF24"
dataset_splits = load_dataset_splits(
  assets("dna/datasets"), transcription_factor, batch_size
)

sample_seq, _ = next(iter(dataset_splits["train"]))
arch_seq_len = sample_seq.shape[1]

torch.manual_seed(42)

model_metrics = {}

for name, model_cls in models.items():
  model_inst = model_cls.to(device)
  optimizer_inst = torch.optim.AdamW(
    model_inst.parameters(), lr=learning_rate
  )
  scheduler_inst = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_inst, T_max=num_steps
  )

  _, _, metrics = train(
    model=model_inst,
    optimizer=optimizer_inst,
    dataset_splits=dataset_splits,
    num_steps=num_steps,
    eval_every=100,
    device=device,
    store_path=assets(f"dna/models/{name}"),
  )
  model_metrics.update({name: metrics})

In [ ]:
# Extract metrics logged per transcription factor.
model_df = []
for model, metrics in model_metrics.items():
  model_df.append(to_df(metrics).assign(model=model))
model_df = pd.concat(model_df)

# Determine order of best performance.
auc_df = model_df[
  (model_df["metric"] == "auc") & (model_df["split"] == "valid")
]
max_auc_by_model = auc_df.groupby("model")["mean"].max()
model_order = max_auc_by_model.sort_values(ascending=False).index.tolist()
model_df["model"] = pd.Categorical(
  model_df["model"], categories=model_order, ordered=True
)

In [ ]:
sns.set_context("notebook", font_scale=1.2, rc={"lines.linewidth": 2.5})
sns.set_style("ticks", {"axes.grid": True})
g = sns.relplot(
  data=model_df,
  x="round",
  y="mean",
  hue="split",
  style="metric",
  kind="line",
  col="model",
  col_order=model_order,
  col_wrap=2,
  alpha=0.8,
  palette=DEFAULT_SPLIT_COLORS,
  dashes=True,
)
g.set_axis_labels("Step", "Value")
g.set(ylim=(0.4, 0.9));

In [ ]:
g = sns.lineplot(
  data=model_df[(model_df["metric"] == "auc")],
  x="round",
  y="mean",
  hue="model",
  style="model",
  alpha=0.8,
)
g.set_xlabel("Step")
g.set_ylabel("auROC");

In [ ]:
print(max_auc_by_model.sort_values(ascending=False))

### 3.5.4. Evaluating on the Test Split


In [ ]:
from dlfb_pytorch.utils.restore import restore

top_model = model_order[0]

model_inst = models[top_model].to(device)
optimizer_inst = torch.optim.AdamW(model_inst.parameters(), lr=learning_rate)

state, _ = restore(
  assets(f"dna/models/{top_model}"),
  model_inst,
  optimizer_inst,
)
model_inst = state["model"]

# Evaluate on the held-out test set.
test_sequences, test_labels = next(iter(dataset_splits["test"]))
metrics = eval_step(model_inst, test_sequences, test_labels, device)
print(metrics)

### 3.5.5. Extensions and Improvements


## 3.6. Summary
